# Classical models

$\renewcommand{\ket}[1]{\left|#1\right\rangle}\renewcommand{\bra}[1]{\left\langle #1\right|}\renewcommand{\braket}[2]{\left\langle #1 \middle| #2 \right\rangle}\renewcommand{\ketbra}[2]{\left|#1\right\rangle\!\left\langle #2\right|}$

This notebook shows how to convert the number of classical MCMC queries, for both the uniform and local spin-flip moves, into wall-clock runtime estimates.

We seek the best classical baseline and therefore benchmark three heterogeneous computing platforms:

* A **sequential processor**. This platform operates at a high clock frequency and provides highly optimized floating-point arithmetic, but offers limited flexibility for parallelizing the energy-difference calculation. The changed terms must therefore be processed mostly sequentially. This corresponds to a high-performance, single-core CPU with a large cache.

* A **spatially parallel processor**. This platform allows the energy-difference calculation to be implemented as a combinational circuit, exposing the available arithmetic parallelism directly in hardware. The reduction depth can scale logarithmically, rather than linearly, in the number of summed terms. This flexibility, however, comes with lower clock frequencies and larger prefactors. This corresponds to an FPGA-based system.

* A **throughput-oriented processor**. This platform can parallelize part of the arithmetic, but it is not naturally suited to accelerating a single MCMC trajectory because the Markov-chain time direction remains sequential. This corresponds to a GPU-based system. The latency of one step of a single chain can be comparable to, or even greater than, that of the CPU implementation because of branching, synchronization, and less efficient scalar control flow. Although the main advantage of a GPU is its throughput across many independent chains or instances, the benchmark used here reports the latency of a single chain to match the sequential query model adopted in the paper.

We select high-end but realistic representatives of these platforms. The CPU implementation runs on an Intel Xeon processor from a DCGP node of the CINECA Leonardo supercomputer. The GPU implementation runs on an NVIDIA H100 NVL GPU. The FPGA implementation targets an AMD Virtex UltraScale+ XCVU19P (-2FSVA3824E).

We estimate the latency of one attempted classical Metropolis proposal as follows.

1. For the CPU implementation, we use a C++ benchmark compiled with aggressive optimization flags for the target architecture. For each tested system size $n$, we generate several SK instances and run a fixed number of attempted Metropolis proposals. The single-step latency is obtained by dividing the total measured time by the total number of attempted proposals. We then fit the forms:

   * $\tau_{\mathrm{CPU}}^{\mathrm{loc}}(n)=a_{\mathrm{CPU}}^{\mathrm{loc}}+b_{\mathrm{CPU}}^{\mathrm{loc}}n$ for the local spin-flip move;
   * $\tau_{\mathrm{CPU}}^{\mathrm{unif}}(n)=a_{\mathrm{CPU}}^{\mathrm{unif}}+b_{\mathrm{CPU}}^{\mathrm{unif}}n+c_{\mathrm{CPU}}^{\mathrm{unif}}n^2$ for the uniform move.

   The difference follows from the cost of calculating the energy difference. A local spin flip changes only the terms involving the flipped spin, so its energy difference requires $O(n)$ work. A uniform move can change every spin, so the dense SK energy must be recomputed, requiring $O(n^2)$ work. The constant terms represent setup and loop overhead. When extrapolating the asymptotic single-query cost, we retain only the size-dependent contribution.

2. For the GPU implementation, we use a CUDA/C++ benchmark implementing the same dense SK Metropolis kernels. One CUDA block is assigned to each Markov chain, and the dense sums are parallelized through block reductions. The reported quantity is the single-chain latency. We fit the same scaling forms as for the CPU, with platform-specific coefficients:

   * $\tau_{\mathrm{GPU}}^{\mathrm{loc}}(n)=a_{\mathrm{GPU}}^{\mathrm{loc}}+b_{\mathrm{GPU}}^{\mathrm{loc}}n$ for the local spin-flip move;
   * $\tau_{\mathrm{GPU}}^{\mathrm{unif}}(n)=a_{\mathrm{GPU}}^{\mathrm{unif}}+b_{\mathrm{GPU}}^{\mathrm{unif}}n+c_{\mathrm{GPU}}^{\mathrm{unif}}n^2$ for the uniform move.

   This model does not include higher-level GPU strategies such as parallel tempering, many-replica execution, or model-specific update schedules.

3. For the FPGA implementation, we write C/C++ HLS kernels and synthesize them with AMD Vitis HLS into circuits mapped to the target FPGA. Even on the high-end FPGA considered here, synthesis is feasible only up to moderate system sizes before the available area becomes limiting. We therefore fit the synthesizable instances and extrapolate their latency to larger $n$, without enforcing the area constraint. This is analogous to the quantum-resource model used elsewhere in this work, where logical resources are counted without requiring the complete device to fit on a single physical chip. For both proposal moves, the fitted latency has logarithmic form:

   * $\tau_{\mathrm{FPGA}}^{\mathrm{loc}}(n)=a_{\mathrm{FPGA}}^{\mathrm{loc}}+b_{\mathrm{FPGA}}^{\mathrm{loc}}\log_2 n$ for the local spin-flip move;
   * $\tau_{\mathrm{FPGA}}^{\mathrm{unif}}(n)=a_{\mathrm{FPGA}}^{\mathrm{unif}}+b_{\mathrm{FPGA}}^{\mathrm{unif}}\log_2 n$ for the uniform move.

The timing measurements are reported in the corresponding benchmark notebooks `2.{2,3,4}_*.ipynb`.

A major difference among these platforms is their arithmetic representation. The CPU and GPU implementations use single-precision floating-point arithmetic, which is the natural and efficient choice on those devices. The FPGA implementation uses fixed-point arithmetic, so the discretization error introduced by finite precision must be accounted for explicitly.
